# ЛР-6: MLlib и robotics datasets

Среда: **Google Colab**.

Полное методическое описание, критерии оценивания и `TEACH CARD` находятся в одноимённом `.md` файле комплекта.

Сквозной пайплайн курса:

$$
\text{Sense} \rightarrow \text{Collect} \rightarrow \text{Stream} \rightarrow
\text{Store} \rightarrow \text{Process} \rightarrow \text{Learn} \rightarrow \text{Teach}
$$


In [ ]:
!pip -q install "pyspark==4.2.0" pandas numpy

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab06_MLlibRobotics")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# 1. Генерация размеченной телеметрии.
rng = np.random.default_rng(7)
N = 12_000

temperature = rng.normal(47, 3.0, N)
vibration_rms = np.abs(rng.normal(0.22, 0.07, N))
battery_v = rng.normal(12.0, 0.35, N)
current_a = np.abs(rng.normal(2.8, 0.8, N))
accel_rms = rng.normal(9.82, 0.18, N)

risk = (
    0.40 * (temperature > 52).astype(float)
    + 0.35 * (vibration_rms > 0.34).astype(float)
    + 0.20 * (battery_v < 11.5).astype(float)
    + 0.15 * (current_a > 4.0).astype(float)
    + rng.normal(0, 0.08, N)
)

label = (risk > 0.42).astype(int)

pdf = pd.DataFrame({
    "temperature": temperature,
    "vibration_rms": vibration_rms,
    "battery_v": battery_v,
    "current_a": current_a,
    "accel_rms": accel_rms,
    "label": label
})

data = spark.createDataFrame(pdf)
train, test = data.randomSplit([0.8, 0.2], seed=42)

# 2. Pipeline: Transformer -> Estimator -> Transformer -> Estimator.
assembler = VectorAssembler(
    inputCols=["temperature", "vibration_rms", "battery_v", "current_a", "accel_rms"],
    outputCol="raw_features"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=80,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, rf])

model = pipeline.fit(train)
pred = model.transform(test)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1 = evaluator_f1.evaluate(pred)
acc = evaluator_acc.evaluate(pred)

print(f"F1       = {f1:.4f}")
print(f"Accuracy = {acc:.4f}")

pred.select("temperature", "vibration_rms", "battery_v", "label", "prediction", "probability").show(15, truncate=False)

rf_model = model.stages[-1]
print("Feature importances:", rf_model.featureImportances)

assert f1 > 0.75

# 3. Необязательная Colab-ячейка исследования современного robotics dataset.
# Выполняется отдельно, т.к. установка LeRobot включает дополнительные зависимости.
#
# !pip -q install "lerobot[dataset]"
# from lerobot.datasets import LeRobotDataset
# robot_ds = LeRobotDataset("lerobot/pusht")
# print("Episodes:", robot_ds.meta.total_episodes)
# print("Frames:", robot_ds.meta.total_frames)
# print("Features:", list(robot_ds.meta.features.keys()))
#
# Официальный Colab LeRobot:
# https://github.com/huggingface/lerobot/blob/main/examples/notebooks/quickstart.ipynb
#
# Официальный Open X-Embodiment Colab:
# https://github.com/google-deepmind/open_x_embodiment/blob/main/colabs/Open_X_Embodiment_Datasets.ipynb


## TEACH CARD

После выполнения кода заполните `TEACH CARD` из `.md`-файла лабораторной работы и приложите его к отчёту.
